PROGETTO 2: SENTIMENT ANALYSIS SU RECENSIONI (KERAS LSTM)

In un progetto reale come facciamo a decidere chi deve scendere in campo?

Se volessimo capire se un cliete è arrabiato o felice ci fidiamo di un lettore che legge una parola alla volta o di uno che legge un intero paragrafo in un istante?

- Sfida tra architetture: reti LSTM e tranformer di BERT
- Anatomia del fallimento, autopsia degli errori
- come rendere questa intelligenza portatile

Confrontiamo i nostri due sfidanti

LSTB vs BERT
Memoria sequenziale contro attenzione globael.
LSTM gestisce le sequenze, è come un trene che percorre i binari, legge una stazione dietro l'altra portando con se un piccolo bagaglio di memoria, è quendi sequenziale. 
BERT invece rivoluziona il contesto. vede la relazioe tra la prima e l'ultima parole in modo istantaneo.

Chi dei due riesce a cogliere il vero cuore del messaggio?

Vantaggi e Limiti
Analisi comparative delle arhitetture
- Le LSTM sono efficienti, richiiedono meno parametri e potenza di calcolo, ideale per dispositivi con risorse limitate o dataset piccoli
- BERT è profondo, la comprensione bidirezionale permette di catturare il senso di una parola basandosi sull'intera frase, superando il limite unidirezionale. Il senso di una parola può cambiuare completamente in base al contesto.
- Tmpo di Training: BERT richiede sessioni di fine-tuning più onerose e l'uso obbligatorio di GPU rispetto alle architetture ricorrenti
- Overfitting: le LSTM tendono a soffrire meno di overfitting su testi brevi, mentre BERT necessità di una regolarizzazione attenta.

Ma come misuriamo chi sta vincendo?
In dataset sbilanciati, l'accuratezza pura può essere un miraggio, per questo usiamo f1-score, giudice imparziale
Ma guardamo anche l'orologio, quanto tempo di mette un modello. In produzione un cliente non aspetta secondi per avere una risposta.

Quanto è profondo il solco tra queste due tecnologie
In genere BERT vince quando le frasi diventano complesse. Nelle frasi dove il significato dipende da parole lontane, BERT comporta un incremento significativo della precisione rispetto a LSTM

Ma il guadagno marginale che otteniamo con il transformers giustifica il maggior costo computazionale? spesso la rispota è si, specialmente se l'errore ha un costo elevato per l'azienda.

Ma anche i giganti cadono
Analisi degli errori

Error Analysis
E' spesso la parte più sottovalutata ma importante.
Guardare solo metriche globali è un errore comune. L'Error Analysis consiste nel posizionare manulmente i casi in cui il modello ha predetto la classe errata.
Identificare pattern negli errori permette di migliorare il preprocessing o di sceglirere un architettura più robusta per certi casi limite.

Quali sono i mostri che spaventano le nostri reti neurali
- Negazione complessa: non è che io non sia soddisfatto. vedere due non, la rete entra in crisi
- Sarcasmo e ironia: il modello fatica a rilevare il tono se le parole usate sono positive ma l intenzione è negativa.
- Ambiguita Lessicale: parole che cambiano significato in base al dominio. (es. 'freddo' in una recensione di un file vs un condizionatore). Senza contesto la rete è cieca
- Fuori vocabolario (OOV) parole gergali o errori di battitura che non sono stati gestiti correttamente dal tokenizer.

Matrice di confuzione Avanzata
Ci dice dove si concentrano gli errori.
Falsi Positivi: recensioni negative che vengono scambiate come positive. Spesso accade quando sono molto parole di elogio usante in modo ironico
Falsi Negativi: casi in cui un utente è soddisfatto ma usa un linguaggio tecnico che il modello intepreta male (magari come freddo e critico)
- Ispezione dei 50 Campioni: estrarre i primi 50 errori per ogni classe e categorizzarli manualmente per trovare la radice del problema.

Diamo una veste matematica a questa analisi dell'errore
Quantificazione dell'Errore
Distribuzione degli scarti
Calcoliamo l'errore totale come la somma delle discrepanze tra le etichette reali e quelle predette su tutot il set di test.
Questa analisi ci aiuta a capire se l'errore è distribuito uniformemente o se è concentrato su una specifica classe di messaggi.
Se l'errore è concentrato su una specifica classe, significa che il modello non ha ancora imparato a discriminare i confini di quella categoria, è il segnale che dobbiamo dare più dati.

Export del Modello
E' il processo finale, lo rendiamo disponibile per l'uso esterno.
Salviamo i pesi ma anche l'intera struttura e il tokenizer, necessario epr processare i nuovo input utentei.

Ma quali sono gli standard industriali per questo passaggio?

Esistono diversi formati di Salvataggio.
* Keras SaveModel: il formato standard di TensorFlow che include l'architettura, pesi e configurazione dell'ottimizzatore.
* Safatensors: formato moderno promosso da Huggin Face per caricare pesi in modo sicure e valoce, evitando l'esecuzione di codice arbitrario
* ONNX: formato interoperabile che permette di eseguire il modello su diversi runtime e linguaggi di programmazione
* Sarialization del Token: salvare il vocabolario e le regole di scomposizione sia per il processo sia come trainin.
Sceglire il formato giusto è il primo passo per un deployd di successo

Pipeline di Caricamento
- Versioning: è fondamentale associare ogni export a una versione specifica.Per tornare indietroin caso di problemi in produzione.
- Compressione: utilizza tecniche di quantizzazione durante l'export del file senza perdere troppa accuratezza.
- Checkpoints: salvare lo stato del modello alla fine di ogni epoca per evitare di perdere ore di lavoro in caso di crash del sistema.

Persistenza dello Stato
Serializzazione dei pesi
E' l'atto di congelare lo stato della mente neurale, prendiamo tutti i pesi ed i bias e li scriviamo in un formato binario, il modello smette di imparare e diventa una funzione deterministica. Per lo stesso input avrò sempre lo stesso output indipendentemente da dove viene eseguito.

In [3]:
"""
================================================================================
FILE: sentiment_comparison_windows.py
================================================================================
DESCRIZIONE: 
Questo script confronta due diverse architetture per la Sentiment Analysis:
1. Una rete LSTM (Long Short-Term Memory) classica costruita in Keras nativo.
2. Un modello BERT (TinyBERT) integrando la libreria Hugging Face Transformers
   direttamente dentro il workflow di Keras 3.

INTERAZIONI PRINCIPALI:
- Keras 3 agisce da orchestratore, usando PyTorch come motore di calcolo (backend).
- Il dataset viene scaricato tramite Hugging Face 'datasets'.
- Per LSTM: Il testo viene convertito in numeri tramite 'TextVectorization' di Keras.
- Per BERT: Il testo viene convertito in token specifici tramite il 'Tokenizer' di Transformers,
  e poi passati al modello BERT che viene "impacchettato" in un layer Keras.
================================================================================
"""

import os
import numpy as np

# --- CONFIGURAZIONE BACKEND ---
# Forza Keras a utilizzare PyTorch per i calcoli tensoriali. 
# Questo permette di mescolare liberamente moduli PyTorch (come BERT) e layer Keras.
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, BertModel

# --- 1. CARICAMENTO DATI ---
def load_real_data():
    """
    Carica il dataset 'Rotten Tomatoes' (recensioni di film).
    
    Flusso:
    1. Scarica i dati via internet (se non presenti in cache).
    2. Estrae un sottoinsieme (subset) per rendere l'esecuzione rapida.
    3. Converte le etichette in array NumPy per la compatibilità con i modelli.
    """
    print("[INFO] Caricamento dataset Rotten Tomatoes...")
    dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")
    
    # .shuffle() assicura che tra i primi 2000 ci sia un mix equo di 0 (NEG) e 1 (POS)
    train_data = dataset["train"].shuffle(seed=42).select(range(2000))
    test_data = dataset["test"].shuffle(seed=42).select(range(500))
    
    # Estraiamo testi e etichette convertendoli in formati standard
    train_texts = list(train_data["text"])
    train_labels = np.array(train_data["label"])
    
    test_texts = list(test_data["text"])
    test_labels = np.array(test_data["label"])
    
    return (train_texts, train_labels), (test_texts, test_labels)

# --- 2. ARCHITETTURA 1: LSTM (Keras Nativo) ---
def build_lstm_model(vocab_size, max_len):
    """
    Crea un modello basato su Recurrent Neural Networks (RNN).
    
    Componenti e Interazioni:
    1. Input: Riceve sequenze di numeri interi (ID delle parole).
    2. Embedding: Trasforma ogni ID in un vettore denso (spazio semantico).
    3. LSTM: Analizza la sequenza nel tempo, mantenendo una "memoria" del contesto.
    4. Dropout: Spegne casualmente dei neuroni per evitare l'overfitting (memorizzazione a memoria).
    5. Dense: Output finale con funzione 'sigmoid' per dare una probabilità 0-1 (Sentimento).
    """
    # Definiamo la forma dell'input (sequenze di lunghezza max_len)
    inputs = layers.Input(shape=(max_len,), name="input_lstm")
    
    # Embedding: 10.000 parole possibili -> vettori da 64 dimensioni
    x = layers.Embedding(input_dim=vocab_size, output_dim=64)(inputs)
    
    # LSTM con 64 unità. return_sequences=False significa che prendiamo solo l'output dell'ultimo timestamp
    x = layers.LSTM(64, return_sequences=False)(x)
    
    # Regolarizzazione
    x = layers.Dropout(0.2)(x)
    
    # Neurone di uscita per classificazione binaria (Positivo/Negativo)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    
    # Assemblaggio del modello Keras
    model = keras.Model(inputs, outputs, name="LSTM_Classifier")
    return model

# --- 3. ARCHITETTURA 2: BERT (Hybrid Keras-Transformers) ---
def build_bert_model(model_ckpt):
    """
    Integra un backbone BERT di Hugging Face in un modello Keras 3.
    Risolve i conflitti di device tra CPU e GPU.
    """
    # Carichiamo il modello encoder
    encoder = BertModel.from_pretrained(model_ckpt)
    
    # Determiniamo il device (CUDA se disponibile, altrimenti CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoder.to(device) # <--- Spostiamo il cuore di BERT sulla GPU

    input_ids = layers.Input(shape=(None,), dtype="int32", name="input_ids")
    attention_mask = layers.Input(shape=(None,), dtype="int32", name="attention_mask")

    def call_bert(inputs):
        ids, mask = inputs
        # IMPORTANTE: Spostiamo i tensori in entrata sullo stesso device del modello
        curr_device = next(encoder.parameters()).device
        ids = ids.to(curr_device)
        mask = mask.to(curr_device)
        
        outputs = encoder(input_ids=ids, attention_mask=mask)
        # Estraiamo il token [CLS]
        return outputs.last_hidden_state[:, 0, :]

    # Definiamo il layer Lambda con output_shape esplicito (128 per TinyBERT)
    x = layers.Lambda(
        call_bert, 
        output_shape=(128,), 
        name="bert_adapter"
    )([input_ids, attention_mask])
    
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="predictions")(x)

    model = keras.Model(inputs=[input_ids, attention_mask], outputs=outputs, name="BERT_Classifier")
    return model

def final_comparison_report(lstm_model, bert_model, x_lstm, x_bert, y_true, texts, num_samples=10):
    """
    Stampa un confronto dettagliato tra i due modelli su un numero definito di campioni.
    
    Args:
        lstm_model: Modello LSTM addestrato.
        bert_model: Modello BERT addestrato.
        x_lstm: Input pre-processati per LSTM.
        x_bert: Lista [input_ids, attention_mask] per BERT.
        y_true: Etichette reali (Ground Truth).
        texts: Testi originali in linguaggio naturale.
        num_samples: Numero di frasi da mostrare.
    """
    print("\n" + "="*80)
    print(" CONFRONTO FINALE: LSTM vs BERT ".center(80, "="))
    print("="*80)

    # Predizioni (otteniamo le probabilità)
    preds_lstm = lstm_model.predict(x_lstm[:num_samples], verbose=0)
    preds_bert = bert_model.predict(x_bert, verbose=0) # x_bert è già filtrato a num_samples se passata correttamente

    for i in range(num_samples):
        # Conversione in label binaria (Soglia 0.5)
        label_lstm = "POS" if preds_lstm[i] > 0.5 else "NEG"
        label_bert = "POS" if preds_bert[i] > 0.5 else "NEG"
        label_true = "POS" if y_true[i] == 1 else "NEG"

        # Evidenziamo gli errori
        status_lstm = "v" if label_lstm == label_true else "x"
        status_bert = "v" if label_bert == label_true else "x"

        print(f"\nFRASE {i+1}: \"{texts[i][:80]}...\"")
        print(f"   > TARGET REALE : {label_true}")
        print(f"   > PREDIZIONE LSTM : {label_lstm} {status_lstm} (prob: {preds_lstm[i][0]:.4f})")
        print(f"   > PREDIZIONE BERT : {label_bert} {status_bert} (prob: {preds_bert[i][0]:.4f})")
        
        if label_lstm != label_bert:
            print("   ! DISACCORDO RILEVATO: BERT e LSTM interpretano diversamente la semantica.")

    print("\n" + "="*80)

# --- 4. FLUSSO DI ESECUZIONE (MAIN) ---

# A. Recupero Dati
(train_texts, train_labels), (test_texts, test_labels) = load_real_data()

# B. Preprocessing per LSTM: Semplice vettorizzazione
# Trasforma: "I love this movie" -> [45, 12, 5, 89, 0, 0...]
MAX_LEN = 50
VOCAB_SIZE = 10000
vectorizer = layers.TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAX_LEN)
vectorizer.adapt(train_texts) # Impara il vocabolario dai testi di training
x_train_lstm = vectorizer(train_texts)

# C. Preprocessing per BERT: Tokenizzazione specifica del modello
# BERT non usa una vettorizzazione standard, ma un sistema a sottoparole (WordPiece)
tokenizer = AutoTokenizer.from_pretrained("google/bert_uncased_L-2_H-128_A-2") # TinyBERT

def bert_tokenize(texts):
    """Converte il testo in tensori PyTorch pronti per BERT."""
    tk = tokenizer(texts, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    return {"input_ids": tk["input_ids"], "attention_mask": tk["attention_mask"]}

x_train_bert = bert_tokenize(train_texts)

# D. Creazione Modelli
lstm_model = build_lstm_model(VOCAB_SIZE, MAX_LEN)
bert_model = build_bert_model("google/bert_uncased_L-2_H-128_A-2")

# E. Compilazione
# LSTM usa parametri standard
lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# BERT richiede un Learning Rate molto piccolo (2e-5) per non distruggere la conoscenza pre-addestrata
bert_model.compile(optimizer=keras.optimizers.Adam(2e-5), loss="binary_crossentropy", metrics=["accuracy"])

# F. Addestramento (1 epoca per dimostrazione)
print("\n[TRAIN] Addestramento LSTM (RNN base)...")
lstm_model.fit(x_train_lstm, train_labels, epochs=5, batch_size=32,validation_split=0.2)

print("\n[TRAIN] Addestramento BERT (Trasformer pre-addestrato)...")
# Nota: passiamo una lista di due input (input_ids e attention_mask) come richiesto dall'architettura BERT
bert_model.fit(
    [x_train_bert["input_ids"], x_train_bert["attention_mask"]], 
    train_labels, 
    epochs=5, 
    batch_size=32
)

# --- 6. EXPORT (Versione Keras 3 / 2026) ---
print("\n[EXPORT] Salvataggio modelli...")

# Formato standard Keras 3 (contiene tutto: architettura e pesi)
lstm_model.save("lstm_sentiment_v1.keras") 

# Per BERT, seguiamo la regola del framework:
# Se vogliamo i pesi in un formato leggibile da Keras:
bert_model.save_weights("bert_weights.weights.h5")

# Se vogliamo esportare in formato Safetensors (Best Practice per Hugging Face)
# Estraiamo i pesi dal modello Keras e salviamoli via torch
import torch
from safetensors.torch import save_file

print("Esportazione pesi BERT in formato Safetensors...")
# Recuperiamo i tensori PyTorch dal modello
tensors = {name: torch.tensor(param) for name, param in zip(bert_model.state_dict().keys(), bert_model.state_dict().values())}
save_file(tensors, "bert_model_final.safetensors")

print(f"\n[SUCCESS] Modelli salvati correttamente!")



# Prepariamo i dati BERT per i primi 10 campioni (importante per il matching)
x_bert_samples = [
    x_train_bert["input_ids"][:10], 
    x_train_bert["attention_mask"][:10]
]

final_comparison_report(
    lstm_model, 
    bert_model, 
    x_train_lstm[:10], 
    x_bert_samples, 
    train_labels[:10], 
    train_texts[:10]
)

[INFO] Caricamento dataset Rotten Tomatoes...


Loading weights: 100%|██████████| 39/39 [00:00<00:00, 3261.97it/s]
[transformers] BertModel LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[TRAIN] Addestramento LSTM (RNN base)...
Epoch 1/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.4956 - loss: 0.6939 - val_accuracy: 0.4975 - val_loss: 0.6933
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.4775 - loss: 0.6946 - val_accuracy: 0.5025 - val_loss: 0.6937
Epoch 3/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.5031 - loss: 0.6937 - val_accuracy: 0.5025 - val_loss: 0.6932
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.5125 - loss: 0.6928 - val_accuracy: 0.5025 - val_loss: 0.6931
Epoch 5/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.5094 - loss: 0.6926 - val_accuracy: 0.5650 - val_loss: 0.6906

[TRAIN] Addestramento BERT (Trasformer pre-addestrato)...
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.5325 - loss: 0.8291
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.5195 - loss: 0.8074
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.5280 - loss: 0.8128
Epoch 4/10
63/

C:\Users\barbara\AppData\Local\Temp\ipykernel_55360\2119406357.py:237: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensors = {name: torch.tensor(param) for name, param in zip(bert_model.state_dict().keys(), bert_model.state_dict().values())}
